# Prepare config

In [1]:
from typing import Any
from pathlib import Path

import yaml

def load_config(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    return config or {}

config = 'configs/gpt2_shakespear_boosting.yaml'
config = load_config(Path(config))
config

{'runtime': {'mode': 'train'},
 'data': {'type': 'shakespeare_char',
  'params': {'dataset_dir': 'data/shakespeare_char',
   'input_file': 'input.txt',
   'train_bin': 'train.bin',
   'val_bin': 'val.bin',
   'meta_file': 'meta.pkl',
   'block_size': 256,
   'batch_size': 32,
   'num_workers': 0,
   'pin_memory': False,
   'val_split': 0.1,
   'download_if_missing': False,
   'reprepare': False}},
 'model': {'type': 'gpt2_boosting_char',
  'params': {'num_learners': 3,
   'shrinkage': 1.0,
   'weak_learner': {'block_size': 256,
    'vocab_size': 65,
    'n_layer': 2,
    'n_head': 6,
    'n_embd': 384,
    'dropout': 0.2,
    'bias': True}}},
 'score': {'type': 'perplexity', 'params': {'metrics': ['perplexity']}},
 'inference': {'enabled': False, 'output_dir': 'outputs/predictions'},
 'training': {'mode': 'boosting',
  'epochs': 2000,
  'eval_step': 2,
  'results_dir': 'results/runs',
  'boosting': {'line_search': {'split': 'val', 'alpha_bounds': [0.0, 1.0]}},
  'optimizer': {'name': '

# Init Trainer

In [2]:
from src.orchestrator.trainer import Trainer

trainer = Trainer(config)

[model-init] weak_learner_params=3672960


In [3]:
trainer.data_module, trainer.model, trainer.score

(<src.data.shakespeare_char.ShakespeareCharDataModule at 0x279ffadaa50>,
 GPT2BoostingLanguageModel(
   (learners): ModuleList(
     (0-2): 3 x GPT(
       (transformer): ModuleDict(
         (wte): Embedding(65, 384)
         (wpe): Embedding(256, 384)
         (drop): Dropout(p=0.2, inplace=False)
         (h): ModuleList(
           (0-1): 2 x Block(
             (ln_1): LayerNorm()
             (attn): CausalSelfAttention(
               (c_attn): Linear(in_features=384, out_features=1152, bias=True)
               (c_proj): Linear(in_features=384, out_features=384, bias=True)
               (attn_dropout): Dropout(p=0.2, inplace=False)
               (resid_dropout): Dropout(p=0.2, inplace=False)
             )
             (ln_2): LayerNorm()
             (mlp): MLP(
               (c_fc): Linear(in_features=384, out_features=1536, bias=True)
               (gelu): GELU(approximate='none')
               (c_proj): Linear(in_features=1536, out_features=384, bias=True)
            

# Start run Trainer

## Load data from data module

In [4]:
from datetime import datetime, timezone

run_start = datetime.now(timezone.utc)
data_config = trainer.config["data"]
data_config

{'type': 'shakespeare_char',
 'params': {'dataset_dir': 'data/shakespeare_char',
  'input_file': 'input.txt',
  'train_bin': 'train.bin',
  'val_bin': 'val.bin',
  'meta_file': 'meta.pkl',
  'block_size': 256,
  'batch_size': 32,
  'num_workers': 0,
  'pin_memory': False,
  'val_split': 0.1,
  'download_if_missing': False,
  'reprepare': False}}

In [5]:
trainer.data_module.load_data()
trainer.data_module.setup_dataloaders()

train_dataloader = trainer.data_module.train_dataloader
val_dataloader = trainer.data_module.val_dataloader

len(train_dataloader), len(val_dataloader)

(31363, 3478)

In [6]:
samples = next(iter(train_dataloader))
samples, samples["input_ids"].shape, samples["targets"].shape

({'input_ids': tensor([[53,  1, 46,  ..., 43, 61, 53],
          [ 1, 58, 46,  ..., 42, 60, 47],
          [13, 30, 17,  ..., 43,  6,  1],
          ...,
          [51, 43, 11,  ..., 33, 25, 21],
          [39, 42, 57,  ..., 50, 43,  1],
          [53, 60, 43,  ..., 21, 17, 32]]),
  'targets': tensor([[ 1, 46, 43,  ..., 61, 53, 51],
          [58, 46, 43,  ..., 60, 47, 57],
          [30, 17, 26,  ...,  6,  1, 61],
          ...,
          [43, 11,  0,  ..., 25, 21, 27],
          [42, 57, 51,  ..., 43,  1, 57],
          [60, 43,  1,  ..., 17, 32, 10]])},
 torch.Size([32, 256]),
 torch.Size([32, 256]))

In [7]:
trainer.data_module.decode(samples['input_ids'][0])

'o her by oath, and the nuptial appointed: between\nwhich time of the contract and limit of the\nsolemnity, her brother Frederick was wrecked at sea,\nhaving in that perished vessel the dowry of his\nsister. But mark how heavily this befell to the\npoor gentlewo'

In [8]:
trainer.data_module.decode(samples['targets'][0])

' her by oath, and the nuptial appointed: between\nwhich time of the contract and limit of the\nsolemnity, her brother Frederick was wrecked at sea,\nhaving in that perished vessel the dowry of his\nsister. But mark how heavily this befell to the\npoor gentlewom'

## Set up model

In [9]:
# Set up training config and device
training_config = trainer.config["training"]
trainer.device = trainer._resolve_device(str(training_config.get("device", "auto")))
# Set up model
trainer.model.setup()
trainer.model.to(trainer.device)

GPT2BoostingLanguageModel(
  (learners): ModuleList(
    (0-2): 3 x GPT(
      (transformer): ModuleDict(
        (wte): Embedding(65, 384)
        (wpe): Embedding(256, 384)
        (drop): Dropout(p=0.2, inplace=False)
        (h): ModuleList(
          (0-1): 2 x Block(
            (ln_1): LayerNorm()
            (attn): CausalSelfAttention(
              (c_attn): Linear(in_features=384, out_features=1152, bias=True)
              (c_proj): Linear(in_features=384, out_features=384, bias=True)
              (attn_dropout): Dropout(p=0.2, inplace=False)
              (resid_dropout): Dropout(p=0.2, inplace=False)
            )
            (ln_2): LayerNorm()
            (mlp): MLP(
              (c_fc): Linear(in_features=384, out_features=1536, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=1536, out_features=384, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
          )
        )
        (ln_f):

## Build skeleton for output format

In [10]:
# Do not allow default training config
training_mode = str(training_config.get("mode", "baseline")).lower()
epochs = int(training_config["epochs"])
eval_step = int(training_config["eval_step"])

run_id = trainer._build_run_id()
results: dict[str, Any] = {
    "metadata": {
        "run_id": run_id,
        "started_at": run_start.isoformat(),
        "finished_at": None,
        "config": trainer.config,
        "data_meta": trainer._collect_data_meta(),
        "score_meta": {
            "metrics": trainer.config.get("score", {}).get("params", {}).get("metrics", ["loss"]),
            "primary_metric": trainer._primary_score_name(),
        },
        "training_mode": training_mode,
    },
    "train": {
        "metadata": {"split": "train", "num_samples": 0},
        "losses": [],
        "scores": {},
        "records": [],
    },
    "val": {
        "metadata": {"split": "val", "num_samples": 0},
        "losses": [],
        "scores": {},
        "records": [],
    },
    "summary": {},
}

In [11]:
results

{'metadata': {'run_id': '20260410_115604_gpt2_boosting_char_shakespeare_char',
  'started_at': '2026-04-10T11:56:03.993002+00:00',
  'finished_at': None,
  'config': {'runtime': {'mode': 'train'},
   'data': {'type': 'shakespeare_char',
    'params': {'dataset_dir': 'data/shakespeare_char',
     'input_file': 'input.txt',
     'train_bin': 'train.bin',
     'val_bin': 'val.bin',
     'meta_file': 'meta.pkl',
     'block_size': 256,
     'batch_size': 32,
     'num_workers': 0,
     'pin_memory': False,
     'val_split': 0.1,
     'download_if_missing': False,
     'reprepare': False}},
   'model': {'type': 'gpt2_boosting_char',
    'params': {'num_learners': 3,
     'shrinkage': 1.0,
     'weak_learner': {'block_size': 256,
      'vocab_size': 65,
      'n_layer': 2,
      'n_head': 6,
      'n_embd': 384,
      'dropout': 0.2,
      'bias': True}}},
   'score': {'type': 'perplexity', 'params': {'metrics': ['perplexity']}},
   'inference': {'enabled': False, 'output_dir': 'outputs/pred

## Start run boosting function

## Curently in process batch function

In [12]:
import torch 

targets = torch.tensor([[1, 2, 3], [4, 5, 6]])
targets.shape

torch.Size([2, 3])

In [13]:
valid_mask = targets.ne(-1)
valid_mask

tensor([[True, True, True],
        [True, True, True]])

In [14]:
safe_targets = targets.masked_fill(~valid_mask, 0)
safe_targets

tensor([[1, 2, 3],
        [4, 5, 6]])

In [15]:
probs = torch.rand(2, 3, 10)  # Example probabilities for a vocab of size 10
gradient = -probs
gradient

tensor([[[-0.4064, -0.3438, -0.3655, -0.1070, -0.9327, -0.5275, -0.9578,
          -0.1780, -0.0863, -0.3197],
         [-0.2079, -0.2015, -0.7757, -0.5133, -0.6090, -0.4912, -0.1502,
          -0.2001, -0.6911, -0.3296],
         [-0.0379, -0.1411, -0.3345, -0.7130, -0.2386, -0.7210, -0.9840,
          -0.7289, -0.7779, -0.9700]],

        [[-0.6934, -0.0789, -0.6319, -0.4856, -0.7004, -0.5937, -0.5613,
          -0.6754, -0.8635, -0.2932],
         [-0.1783, -0.8889, -0.3005, -0.8255, -0.9343, -0.1870, -0.4844,
          -0.8843, -0.6856, -0.0621],
         [-0.7478, -0.5961, -0.3475, -0.6985, -0.0888, -0.3990, -0.2854,
          -0.2720, -0.9681, -0.1570]]])

In [16]:
gradient.scatter_add_(
    dim=-1,
    index=safe_targets.unsqueeze(-1),
    src=valid_mask.unsqueeze(-1).to(gradient.dtype),
)

tensor([[[-0.4064,  0.6562, -0.3655, -0.1070, -0.9327, -0.5275, -0.9578,
          -0.1780, -0.0863, -0.3197],
         [-0.2079, -0.2015,  0.2243, -0.5133, -0.6090, -0.4912, -0.1502,
          -0.2001, -0.6911, -0.3296],
         [-0.0379, -0.1411, -0.3345,  0.2870, -0.2386, -0.7210, -0.9840,
          -0.7289, -0.7779, -0.9700]],

        [[-0.6934, -0.0789, -0.6319, -0.4856,  0.2996, -0.5937, -0.5613,
          -0.6754, -0.8635, -0.2932],
         [-0.1783, -0.8889, -0.3005, -0.8255, -0.9343,  0.8130, -0.4844,
          -0.8843, -0.6856, -0.0621],
         [-0.7478, -0.5961, -0.3475, -0.6985, -0.0888, -0.3990,  0.7146,
          -0.2720, -0.9681, -0.1570]]])